# crm_sync — create client interactions from Python

One function writes one row into the dashboard:

```python
create_client_engagement(
    external_client="Acme Retirement Trust",
    intake_type="serf",
    project_type="Data Request",
)
```

That is the whole thing. `CrmSync` below is the same job for a batch, with idempotency and
per-record failure isolation.

There is nothing to configure in a checkout — `SQLITE_DIR` is read from the environment or
from the same `.env` the Next.js app uses. There is no API key and no password, because
there is no network boundary to authenticate across: this opens the SQLite file directly,
so the access control is the folder's permissions.

> **Anything you create here is real.** It appears in the dashboard immediately, for
> everyone. Every cell that writes is guarded by `DRY_RUN`, and the demo rows are stamped
> with the marker below so you can find and delete them afterwards.

In [ ]:
from crm_sync import ClientInteraction, CrmSync, create_client_engagement, load_config
from crm_sync.core.exceptions import ConfigError, CrnRequiredError, DashboardVisibilityError
from crm_sync.db.registries import Registries

DRY_RUN = True                                    # <- flip to False when you mean it
NOTEBOOK_MARKER = "NOTEBOOK DEMO - DELETE ME"     # so you can find these rows later

cfg = load_config()
cfg.ensure_ready()          # raises with a fix-it message if SQLITE_DIR is wrong

print("database    :", cfg.sqlite_dir)
print("engagements :", cfg.engagements_db.name)
print("writing as  :", cfg.bot_display_name)

# Typical output
# --------------
# database    : D:\Data\CRM
# engagements : engagements.sqlite
# writing as  : CRM Sync
#
# ConfigError here means SQLITE_DIR is unset or the .sqlite files are missing — start the
# app once (npm run dev) or run npm run seed, and the files appear.

## What am I allowed to pass?

Intake types, project types, departments and teams are all **managed in the dashboard**, so
the valid values are whatever this database says they are right now — not whatever was true
when this notebook was written.

Pass the **role token** rather than the display name where one exists: `"serf"` keeps
working after someone renames the type to "Service Request", and `"SERF"` does not.

In [ ]:
reg = Registries.load(cfg)          # read-only; safe against a live database

print("intake types :")
for key, (display, role) in reg.intake_types.items():
    print(f"    {display:<20} role token: {role or '(none)'}")

print("\nproject types:", ", ".join(sorted(d for d, _ in reg.project_types.values())))
print("departments  :", ", ".join(sorted(reg.departments.values())))
print("teams        :", ", ".join(sorted(reg.teams.values())))
print("active people:", len(reg.active_members))

# Typical output
# --------------
# intake types :
#     IRQ                  role token: irq
#     SERF                 role token: serf
#     Ad-Hoc               role token: ad_hoc
#
# project types: Data Request, Data Update, Discovery Meeting, Follow-up Material,
#                Follow-up Meeting, Meeting, Other, PCR
# departments  : Advisory, Brokerage, Institutional, Retirement
# teams        : Default Team
# active people: 13
#
# A value outside these lists is refused before anything is written — an intake type the
# dashboard does not know would produce a row that renders as blank.

## One interaction

`create_client_engagement` is the whole single-row path. It validates, resolves the client,
writes, and returns the new engagement's id.

In [ ]:
if not DRY_RUN:
    engagement_id = create_client_engagement(
        external_client=f"{NOTEBOOK_MARKER} Acme Retirement Trust",
        intake_type="serf",
        project_type="Data Request",
    )
    print("created engagement", engagement_id)
else:
    print("DRY_RUN is True — nothing written.")
    print("Set DRY_RUN = False in the setup cell at the top, then re-run this cell.")

# Typical parameters — all eight, three of which are required
# ----------------------------------------------------------
# create_client_engagement(
#     external_client="Acme Retirement Trust",   # required; the end client
#     intake_type="serf",                        # required; role token or display name
#     project_type="Data Request",               # required; display name
#     internal_client="Acme 401k",               # optional; blank if unregistered
#     date_started=datetime(2026, 4, 2),         # optional; defaults to today
#     date_finished="2026-04-09",                # optional; "" means NULL, still open
#     crn="CRN-000042",                          # optional; see below
#     project_id="PRJ-1042",                     # optional; blank => NULL
# )
#
# What the CRN does depends on what already exists, and the precedence is worth knowing:
#   - client already registered   -> its stored CRN wins; the one you pass is ignored
#   - CRN belongs to another name -> that client wins, and yours is treated as an alias
#   - neither                     -> a new client is registered under the CRN you passed
#   - no CRN at all               -> a PENDING- placeholder, flagged in the dashboard
#
# Typical output
# --------------
# created engagement 4187
#
# The row lands with status "In Progress", no team and no members — it shows up unassigned,
# with a yellow badge, for someone to pick up. That is deliberate: a bot cannot know whose
# work this is, and guessing would put it in the wrong person's queue.

## What goes wrong, and what it means

| Exception | What happened | What to do |
|---|---|---|
| `ValueError` | an unregistered intake or project type, a blank client, a malformed CRN, an unparseable date | check the registry cell above |
| `ConfigError` | `SQLITE_DIR` is unset, or the `.sqlite` files are missing | start the app once, or `npm run seed` |
| `CrnRequiredError` | policy requires a CRN and none was supplied or derivable | pass `crn=` |
| `DashboardVisibilityError` | the row committed but will not render — a post-write check caught it | read the message; it names the column |

Every validation runs **before** a connection is opened, so a rejected call writes nothing
at all. The cell below proves it against the live database.

In [ ]:
try:
    create_client_engagement(
        external_client=f"{NOTEBOOK_MARKER} Never Created",
        intake_type="NOT_AN_INTAKE_TYPE",
        project_type="Data Request",
    )
except ValueError as exc:
    print("ValueError:", exc)

# Typical output
# --------------
# ValueError: intake_type='NOT_AN_INTAKE_TYPE' is not a valid intake type. Valid options:
# Ad-Hoc, IRQ, SERF. Role tokens: ad_hoc, irq, serf.
#
# Nothing was written — not a partial row, not a client record. Safe to run as often as
# you like.

## Many interactions

`CrmSync` is the batch path. Over a loop of `create_client_engagement` calls it adds four
things worth having:

- **one connection** for the whole run rather than one per row;
- **failure isolation** — a bad record is recorded in the summary and the rest continues;
- **`dry_run()`** — the full validation matrix against live registries, writing nothing;
- **idempotency** — give a record a `dedupe_key` and re-running the job will not duplicate
  it, which is what makes a scheduled import safe to retry.

Only five fields are required. `team` and `team_members` are deliberately left empty: the
dashboard renders those rows as unassigned, which is the honest state for something a bot
created.

In [ ]:
def fetch_records():
    """Your source goes here — a queue, an export, a spreadsheet."""
    yield ClientInteraction(
        client_name=f"{NOTEBOOK_MARKER} Acme Retirement Trust",
        internal_client_name="Acme 401k",
        internal_client_dept="Retirement",
        intake_type="serf",
        project_type="Data Request",
        client_crn="CRN-000042",
        dedupe_key="notebook-demo:1",
    )
    yield ClientInteraction(
        client_name=f"{NOTEBOOK_MARKER} Globex Pension",
        internal_client_name="Globex DB Plan",
        internal_client_dept="Institutional",
        intake_type="irq",
        project_type="Meeting",
        nna=250_000,
        dedupe_key="notebook-demo:2",
    )

print(sum(1 for _ in fetch_records()), "records ready")

# Typical parameters — the five required fields, then the useful optional ones
# ---------------------------------------------------------------------------
# ClientInteraction(
#     client_name="Acme Retirement Trust",     # required
#     internal_client_name="Acme 401k",        # required
#     internal_client_dept="Retirement",       # required; must be a managed department
#     intake_type="serf",                      # required; role token preferred
#     project_type="Data Request",             # required
#
#     client_crn="CRN-000042",                 # else a PENDING- placeholder is assigned
#     status="In Progress",                    # In Progress | Awaiting Meeting | Follow Up | Completed
#     date_started="2026-04-02",               # defaults to today
#     date_finished=None,                      # None while still open
#     nna=250_000,                             # net new assets, in dollars
#     tickers_mentioned=["AAPL", "MSFT"],      # Ad-Hoc only
#     notes="Called about the Q1 rebalance.",
#     dedupe_key="intake-queue:47182",         # a stable id from YOUR system
# )
#
# dedupe_key is the one to think about. Use whatever id the source system already has —
# then a re-run after a half-finished job skips what landed instead of duplicating it.

In [ ]:
with CrmSync.from_env() as sync:
    summary = sync.dry_run(fetch_records())      # prints its own summary — see below

print("seen", summary.total, "| would write", summary.written, "| failed", summary.failed)

# Everything above that last line goes to sys.stderr, which Jupyter paints red and which
# reads like a crash. It is not — it is the progress log and the summary. dry_run() always
# prints it; run_batch() takes print_summary=False, which the next cell uses so the output
# comes back as ordinary text.
#
# Typical output
# --------------
# [INFO ] run started
# [INFO ] record started  (dryrun-1)
# [INFO ] team: No team: this interaction lands in the global unassigned inbox ...
# [INFO ] team_members: No assignees: renders with the yellow 'Unassigned' badge ...
#
# ==================================================================
# crm_sync batch summary
# ==================================================================
#   records seen : 2
#   written      : 2
#   deduped      : 0  (already present, skipped)
#   failed       : 0
#   findings     : unassigned_roster=2, unassigned_team=2
#   exit code    : 0
# ==================================================================
# DRY RUN - nothing was written.
#
# seen 2 | would write 2 | failed 0
#
# Those two findings are informational and expected: these records name no team and no
# assignee, so they render as unassigned. That is the intended state for bot-created work,
# not something to fix.
#
# `written` on a dry run means "would have been written". Validation ran against the live
# registries, so a department someone has since renamed fails here rather than at 3am.

In [ ]:
if not DRY_RUN:
    with CrmSync.from_env() as sync:
        summary = sync.run_batch(fetch_records(), print_summary=False)
    print(summary.render())
    print("exit code:", summary.exit_code)
    if summary.failures:
        for correlation_id, message in summary.failures.items():
            print(f"  {correlation_id}: {message}")
else:
    print("DRY_RUN is True — nothing written.")

# Exit codes, for a scheduler
# ---------------------------
#   0  clean
#   1  some records failed        <- the rest were still written
#   2  could not start            <- config or database problem
#   3  everything failed
#
# In a script: raise SystemExit(summary.exit_code)
#
# CrmSync.from_env() takes any config option as an override:
#   CrmSync.from_env(strict=False)              write records that only produced warnings
#   CrmSync.from_env(verify_after_write=False)  skip the post-write visibility check
#   CrmSync.from_env(log_dir="C:/logs")         write crm_sync.jsonl and alerts.log there

## Cleaning up, and where to go next

**To remove what this notebook wrote**, search the dashboard for `NOTEBOOK DEMO - DELETE ME`
and delete those interactions. They are ordinary rows; nothing special is needed.

**Alert sinks** push failures somewhere you will see them. A console sink is wired by
default; add your own:

```python
from crm_sync.utils.monitoring import WebhookAlertSink, CallableAlertSink

sync = CrmSync.from_env()
sync.add_alert_sink(WebhookAlertSink("https://hooks.example.com/crm"))
sync.add_alert_sink(CallableAlertSink(lambda alert: log.error(alert.message)))
```

**Live updates.** Set `SYNC_NUDGE_SECRET` and `CRM_BASE_URL` and a write nudges the running
app over Server-Sent Events, so open dashboards show the new row without a refresh. Without
the secret the nudge is skipped and everything else still works.

**Next:**

- `backend/crm_sync/tests/example_job.py` — a batch job template written to be copied
- `backend/crm_sync/docs/README.md` — the full account, including every validation rule and
  why it exists
- `python -m crm_sync.test` — the smoke test; writes a row, checks it, deletes it
- `backend/notebooks/portfolio_data.ipynb` — the other half, for portfolio analytics